Takes the gdas deeplinks export spreadsheet and creates MARC files

In [1]:
import csv
import datetime
from pymarc import Record, Field, Subfield, Indicators, Leader
from pymarc import TextWriter

In [ ]:
# reads in the gdas import csv and writes out a mrk file

writer = TextWriter(open('out/new-uploads.mrk','wt'))
with open('data/new-upload.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        record = Record()
        record.leader = Leader("     nem a22     5u 4500")
        record.add_field(Field('005  '+datetime.datetime.now().strftime("%Y%m%d%H%M%S")+'.0'))
        record.add_field(Field('007  aj |||||'))
        record.add_field(Field('007  cr c|||||a|na|'))
        if row['Publishing_Date']:
            record.add_field(Field('008  260202s'+row['Publishing_Date']+' xxk|||| z |s 0 eng d'))
        record.add_field(Field(tag = '034', indicators = Indicators('0','#'), 
              subfields = [ Subfield(code='a', value='a'),
                           Subfield(code='d', value=row['Bounding_Box_W']), 
                           Subfield(code='e', value=row['Bounding_Box_E']), 
                           Subfield(code='f', value=row['Bounding_Box_N']), 
                           Subfield(code='g', value=row['Bounding_Box_S']) ] ))
        if row['ISBN']:
            record.add_field(Field(tag = '035', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='(uk)'+row['ISBN'])]))
        record.add_field(Field(tag = '040', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='uk'),Subfield(code='b', value='eng'),Subfield(code='c', value='uk')]))
        record.add_field(Field(tag = '052', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='uk'),Subfield(code='b', value='eng')]))
        record.add_field(Field('110', indicators = Indicators('2','#'), subfields = [Subfield(code='a', value=row['Author']),Subfield(code='e', value='cartographer')]))      
        record.add_field(Field(tag = '245', indicators = Indicators('1','0'), subfields = [Subfield(code='a', value=row['Title']), Subfield(code='h', value='[electronic resource] /'), Subfield(code='c', value=row['Publisher']+'.')])) 
        if row['Edition']:
            record.add_field(Field(tag = '250', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value=row['Edition'])]))           
        if row['Projection']:
            proj = row['Projection']
            if 'OSGB' in proj:
               record.add_field(Field('255', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='Scales vary'),Subfield(code='b', value='British National Grid'),Subfield(code='c', value=row['Bounding_Box_255'])]))  
            elif 'TM65' in proj:
                record.add_field(Field('255', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='Scales vary'),Subfield(code='b', value='TM65 / Irish Grid'),Subfield(code='c', value=row['Bounding_Box_255'])])) 
            elif 'WGS84' in proj:
                record.add_field(Field('255', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='Scales vary'),Subfield(code='b', value='WGS84 Antarctic Polar Stereographic'),Subfield(code='c', value=row['Bounding_Box_255'])])) 
            elif 'Web' in proj:
                record.add_field(Field('255', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='Scales vary'),Subfield(code='b', value='Web Mercator'),Subfield(code='c', value=row['Bounding_Box_255'])]))     
        if row['Place_of_publication']:
            record.add_field(Field(tag = '264', indicators = Indicators('#','1'), subfields = [Subfield(code='a', value=row['Place_of_publication']),Subfield(code='b', value=row['Publishing_Date']),Subfield(code='c', value=row['marcPublisher'])]))
        if row['Extent']:
            record.add_field(Field(tag = '300', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value=row['Extent'])]))  
        if row['Representation'] =='Vector': 
            record.add_field(Field(tag = '336', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='cartographic dataset'),Subfield(code='2', value='rdacontent')]))
        elif row['Representation'] =='Raster': 
            record.add_field(Field(tag = '336', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='cartographic image'),Subfield(code='2', value='rdacontent')]))      
        record.add_field(Field(tag = '337', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='computer'),Subfield(code='2', value='rdamedia')]))
        record.add_field(Field(tag = '338', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='online resource'),Subfield(code='2', value='2rdacarrier')]))
        record.add_field(Field(tag = '342', indicators = Indicators('0','0'), subfields = [Subfield(code='b', value='degree'),Subfield(code='c', value='0.0174532925199433'),Subfield(code='d', value='0.0174532925199433')]))
        if row['Projection']:
            proj = row['Projection']
            if 'OSGB' in proj:
                record.add_field(Field(tag = '342', indicators = Indicators('0','1'), subfields = [Subfield(code='a', value='Transverse'), Subfield(code='g', value='-2'),Subfield(code='h', value='49'), Subfield(code='i', value='400,000'),Subfield(code='j', value='-100,000'),Subfield(code='k', value='0.9996012717 - British National Grid')]))
                record.add_field(Field(tag = '342', indicators = Indicators('0','2'), subfields = [Subfield(code='a', value='OSGB'),Subfield(code='b', value='metre'),Subfield(code='c', value='1'),Subfield(code='c', value='1'),Subfield(code='d', value='1'),Subfield(code='p', value='British National Grid')]))
                record.add_field(Field(tag = '342', indicators = Indicators('0','5'), subfields = [Subfield(code='a', value='OSGB_1936'),Subfield(code='q', value='Airy'),Subfield(code='r', value='6377563.396'),Subfield(code='s', value='299.3249646')]))
            elif 'TM65' in proj:
                record.add_field(Field(tag = '342', indicators = Indicators('0','1'), subfields = [Subfield(code='a', value='Transverse'), Subfield(code='g', value='-2'),Subfield(code='h', value='49'), Subfield(code='i', value='400,000'),Subfield(code='j', value='-100,000'),Subfield(code='k', value='0.9996012717 - British National Grid')]))
                record.add_field(Field(tag = '342', indicators = Indicators('0','2'), subfields = [Subfield(code='a', value='TM65'),Subfield(code='b', value='metre'),Subfield(code='c', value='1'),Subfield(code='c', value='1'),Subfield(code='d', value='1'),Subfield(code='p', value='Irish Grid')]))
                record.add_field(Field(tag = '342', indicators = Indicators('0','5'), subfields = [Subfield(code='a', value='TM65'),Subfield(code='q', value='Airy'),Subfield(code='r', value='6377340.189'),Subfield(code='s', value='299.3249646')]))
            elif 'WGS84' in proj:
                record.add_field(Field(tag = '342', indicators = Indicators('0','1'), subfields = [Subfield(code='a', value='EPSG:3031')]))
            elif 'Web' in proj:
                record.add_field(Field(tag = '342', indicators = Indicators('0','1'), subfields = [Subfield(code='a', value='Transverse'), Subfield(code='g', value='-2'),Subfield(code='h', value='49'),Subfield(code='i', value='400,000'),Subfield(code='j', value='-100,000'),Subfield(code='k', value='0.9996012717 - Web Mercator')]))
        if row['Geographic_name']:
            gname = row['Geographic_name']
            if gname == 'Great Britain':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01204623')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)6618604')])) 
            elif gname == 'Northern Ireland':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01205215')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)2641364')])) 
            elif gname == 'Scotland':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01206715')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)2638360')]))     
            elif gname == 'Wales':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01207649')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)2634894')]))     
            elif gname == 'England':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01219920')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)6269131')])) 
            elif gname == 'Antarctica':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='fast'),Subfield(code='0', value='(OCoLC)fst01239992')]))  
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)6269204')])) 
            elif gname == 'UK':
                record.add_field(Field(tag = '651', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=gname),Subfield(code='2', value='geonames'),Subfield(code='0', value='(geonames)2635167')])) 
            else:
                record.add_field(Field(tag = '651', indicators = Indicators('#','0'), subfields = [Subfield(code='a', value=gname)])) 
        record.add_field(Field(tag = '655', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=row['Cartographic_genre']),Subfield(code='2', value='lcgft')]))
        if row['Cartographic_genre_2']:
            record.add_field(Field(tag = '655', indicators = Indicators('#','7'), subfields = [Subfield(code='a', value=row['Cartographic_genre_2']),Subfield(code='2', value='lcgft')]))
        record.add_field(Field(tag = '856', indicators = Indicators('4','0'), subfields = [Subfield(code='u', value='https://ldl.themapcloud.com/mapping?epsg='+row['epsgCode']+'&datasetName='+row['dataset-name']+'&snapshotYear='+row['snapshot-year'])]))
        writer.write(record)
writer.close()

In [15]:
with open("out/marcrecords.mrk", "r") as file:
    mrk = file.read().replace("  \\", "")
with open("out/marcrecords-ed.mrk", "w") as file:
    file.write(mrk)

In [ ]:
# reads in the csv and writes out a .mrc file
# file will not validate in marc report


with open('out/gdas-mrc.mrc', 'wb') as out:
    with open('data/Deeplinks_UAT.csv', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            record = Record()
            record.leader = Leader("     nem a22     5u 4500")
            record.add_field(Field('005  '+datetime.datetime.now().strftime("%Y%m%d%H%M%S")+'.0'))
            record.add_field(Field('007  aj |||||'))
            record.add_field(Field('007  cr c|||||a|na|'))
            if row['F008_field']:
                record.add_field(Field('008  260202'+row['F008_field']))
            record.add_field(Field(tag = '034', indicators = Indicators('0','#'), 
                subfields = [ Subfield(code='a', value='a'),
                            Subfield(code='d', value=row['034_d']), 
                            Subfield(code='e', value=row['034_e']), 
                            Subfield(code='f', value=row['034_f']), 
                            Subfield(code='g', value=row['034_g']) ] ))
            if row['referenceId']:
                record.add_field(Field(tag = '035', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='(uk)'+row['referenceId'])]))
            record.add_field(Field(tag = '040', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='uk'),Subfield(code='b', value='eng'),Subfield(code='c', value='uk')]))
            record.add_field(Field(tag = '052', indicators = Indicators('#','#'), subfields = [Subfield(code='a', value='uk'),Subfield(code='b', value='eng')]))
            record.add_field(Field('110  '+row['F110']))
            record.add_field(
            Field(tag = '245',
                indicators = Indicators('1','0'),
                subfields = [
                    Subfield(code='a', value=row['marcTitle_245a']),
                    Subfield(code='h', value='[electronic resource] /'),
                    Subfield(code='c', value=row['marcPublisher']+'.')
                    ]))
            if row['F250a_edition_']:
                record.add_field(Field(tag = '250', indicators = Indicators('#','#'),
                        subfields = [Subfield(code='a', value=row['F250a_edition_'])]))
            if row['F255']:
                record.add_field(Field('255  ##'+row['F255']))
            if row['F264_1']:
                record.add_field(Field('264  #1'+row['F264_1'].replace(':$b',' :$b')))
            record.add_field(Field('300  ##$a1 geospatial database'))
            record.add_field(Field('336  ##$acartographic image$2rdacontent'))
            record.add_field(Field('336  ##$acartographic dataset$2rdacontent'))
            record.add_field(Field('337  ##$acomputer$2rdamedia'))
            record.add_field(Field('338  ##$aonline resource$2rdacarrier'))
            if row['F342_00']:
                record.add_field(Field('342  00'+row['F342_00']))
            if row['F342_01']:
                record.add_field(Field('342  01'+row['F342_01']))
            if row['F342_02']:
                record.add_field(Field('342  02'+row['F342_02']))
            if row['F342_05']:
                record.add_field(Field('342  05'+row['F342_05']))
            if row['F651']:
                record.add_field(Field('651  #0'+row['F651'].replace('\\','$').replace('$0','').replace('$a ','$a')))
            if row['F651_1']:
                record.add_field(Field('651  #0'+(row['F651_1'].replace('\\','$').replace('$a ','$a'))))
            if row['F651_2']:
                record.add_field(Field('651  #7'+row['F651_2'].replace('\\','$').replace('$z ','$z').replace('$a ','$a').replace('$2 ','$2').replace('0 (','0(')))
            # if row['F651_3']:
            #     record.add_field(Field('651  #7'+row['F651_3'].replace('\\','$').replace('$a ','$a')))
            if row['F655_1']:
                record.add_field(Field('655  #7'+row['F655_1'].replace('\\','$').replace('$a ','$a').replace('$2 ','$2')))
            if row['F655_2']:
                record.add_field(Field('655  #7'+row['F655_2'].replace('\\','$').replace('$a ','$a').replace('$2 ','$2')))
            if row['F655_3']:
                record.add_field(Field('655  #7'+row['F655_3'].replace('\\','$').replace('$a ','$a').replace('$2 ','$2')))
            if row['Deeplinks URL']:
                record.add_field(Field('856  40$u'+row['Deeplinks URL']))
            out.write(record.as_marc())       
        